# ETL — Coleta e Preparação de Dados

Este notebook realiza o processo de **ETL (Extract, Transform, Load)** para as criptomoedas utilizadas na análise de carteiras de investimento.

O período de extração é fixado entre `2020-01-01` e `2025-12-31`, alinhado exatamente ao intervalo utilizado no notebook de análise exploratória da carteira.

| Símbolo | Nome     |
|---------|----------|
| BTC     | Bitcoin  |
| ETH     | Ethereum |
| XRP     | Ripple   |
| DASH    | Dash     |

## Bibliotecas

Importação das bibliotecas necessárias para coleta e manipulação dos dados.

In [2]:
import pandas as pd
import yfinance as yf
from datetime import date

## Coleta de dados

Os dados são extraídos diretamente da API do Yahoo Finance via `yfinance`, com granularidade diária (`1d`).

O intervalo de datas é definido explicitamente para garantir consistência com o período analisado no notebook de análise exploratória.

In [3]:
inicio = date(2020, 1, 1)
fim    = date(2025, 12, 31)

btc  = yf.Ticker("BTC-USD").history(start=inicio, end=fim, interval="1d")
eth  = yf.Ticker("ETH-USD").history(start=inicio, end=fim, interval="1d")
xrp  = yf.Ticker("XRP-USD").history(start=inicio, end=fim, interval="1d")
dash = yf.Ticker("DASH-USD").history(start=inicio, end=fim, interval="1d")

## Limpeza e transformação

A função `limpar_df` aplica as seguintes transformações em cada DataFrame:

- Reseta o índice, tornando a data uma coluna comum
- Padroniza o nome da coluna de data para `date`
- Converte a coluna `date` para o tipo `datetime`
- Remove colunas irrelevantes (`Dividends`, `Stock Splits`)
- Remove linhas com valores nulos

In [4]:
def limpar_df(df):
    df = df.copy()

    # resetar index (data vira coluna)
    df = df.reset_index()

    # garantir nome padrão
    df.rename(columns={'Date': 'date'}, inplace=True)

    # converter pra timestamp
    df['date'] = pd.to_datetime(df['date'])

    # remover colunas desnecessárias
    colunas_remover = ['Dividends', 'Stock Splits']
    df = df.drop(columns=[c for c in colunas_remover if c in df.columns])

    # remover nulos
    df = df.dropna()

    return df

In [5]:
btc  = limpar_df(btc)
eth  = limpar_df(eth)
xrp  = limpar_df(xrp)
dash = limpar_df(dash)

btc.head()

,date,Open,High,Low,Close,Volume
0,2020-01-01 00:00:00+00:00,7194.892090,7254.330566,7174.944336,7200.174316,18565664997
1,2020-01-02 00:00:00+00:00,7202.551270,7212.155273,6935.270020,6985.470215,20802083465
2,2020-01-03 00:00:00+00:00,6984.428711,7413.715332,6914.996094,7344.884277,28111481032
3,2020-01-04 00:00:00+00:00,7345.375488,7427.385742,7309.514160,7410.656738,18444271275
4,2020-01-05 00:00:00+00:00,7410.451660,7544.497070,7400.535645,7411.317383,19725074095


## Exportação

Os DataFrames limpos são exportados em formato CSV, um arquivo por criptomoeda, sem o índice numérico do pandas.

In [ ]:
btc.to_csv('../data/cripto/BTC.csv',   index=False)
eth.to_csv('../data/cripto/ETH.csv',   index=False)
xrp.to_csv('../data/cripto/XRP.csv',   index=False)
dash.to_csv('../data/cripto/DASH.csv', index=False)